In [16]:
!uv pip install chromadb anthropic firebase-admin
!uv pip install voyageai

Using Python 3.12.3 environment at: /Users/sagar/dev/TheBrain/.venv
Checked 3 packages in 7ms
Using Python 3.12.3 environment at: /Users/sagar/dev/TheBrain/.venv
Resolved 55 packages in 1.18s                                        
Prepared 20 packages in 238ms                                            
Installed 20 packages in 32ms                               
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.13.5
 + aiolimiter==1.2.1
 + aiosignal==1.4.0
 + ffmpeg-python==0.2.0
 + frozenlist==1.8.0
 + future==1.0.0
 + jsonpatch==1.33
 + langchain-core==1.3.0
 + langchain-text-splitters==1.1.2
 + langsmith==0.7.33
 + multidict==6.7.1
 + pillow==12.2.0
 + propcache==0.4.1
 + requests-toolbelt==1.0.0
 + uuid-utils==0.14.1
 + voyageai==0.3.7
 + xxhash==3.6.0
 + yarl==1.23.0
 + zstandard==0.25.0


In [3]:
import json
import os
import chromadb
import anthropic
import firebase_admin
from firebase_admin import credentials, firestore

# Path to your service account key
SERVICE_ACCOUNT_PATH = "../credentials/our-kitchen-app-5557e-firebase-adminsdk-fbsvc-7b99d2edb7.json"

# Voyage key from secrets.json (new)
with open("/Users/sagar/dev/TheBrain/secrets.json") as f:
    secrets = json.load(f)
    
client = anthropic.Anthropic(api_key=secrets["ANTHROPIC_API_KEY"])
voyage_client = voyageai.Client(api_key=secrets["VOYAGE_API_KEY"])

# Init Firebase (reuse your existing pattern from Phase 1)
if not firebase_admin._apps:
    cred = credentials.Certificate(SERVICE_ACCOUNT_PATH)
    firebase_admin.initialize_app(cred)

db = firestore.client()
client = anthropic.Anthropic()

print("Firebase and Anthropic clients ready.")

Firebase and Anthropic clients ready.


In [9]:
def fetch_meals():
    meals_ref = db.collection("meals")
    docs = meals_ref.stream()
    
    meals = []
    for doc in docs:
        data = doc.to_dict()
        data["id"] = doc.id
        meals.append(data)
    
    print(f"Fetched {len(meals)} meals from Firestore.")
    return meals

meals = fetch_meals()

# Preview one meal so you can see the shape of your data
print(json.dumps(meals[0], indent=2, default=str))

Fetched 11 meals from Firestore.
{
  "id": "1774991541622",
  "ts": 1774991541622,
  "cuisine": "Vietnamese",
  "rating": 5,
  "note": "",
  "proteins": [
    "Chicken"
  ],
  "ingredients": [
    {
      "qty": "2 lbs",
      "name": "chicken breast"
    },
    {
      "qty": "4 rolls",
      "name": "Vietnamese baguettes"
    },
    {
      "qty": "1/2 cup",
      "name": "mayonnaise"
    },
    {
      "qty": "2 medium",
      "name": "cucumber"
    },
    {
      "qty": "1 cup",
      "name": "pickled daikon radish"
    },
    {
      "qty": "1 cup",
      "name": "pickled carrots"
    },
    {
      "qty": "1 bunch",
      "name": "fresh cilantro"
    },
    {
      "qty": "2 whole",
      "name": "jalape\u00f1o peppers"
    },
    {
      "qty": "3 tbsp",
      "name": "soy sauce"
    }
  ],
  "name": "Bahn Mi Subs",
  "fav": false,
  "time": ""
}


In [11]:
# Inspect the raw shape of your data
for meal in meals[:3]:
    print(meal.get("name"))
    print(type(meal.get("ingredients")))
    print(meal.get("ingredients"))
    print("---")

Bahn Mi Subs
<class 'list'>
[{'qty': '2 lbs', 'name': 'chicken breast'}, {'qty': '4 rolls', 'name': 'Vietnamese baguettes'}, {'qty': '1/2 cup', 'name': 'mayonnaise'}, {'qty': '2 medium', 'name': 'cucumber'}, {'qty': '1 cup', 'name': 'pickled daikon radish'}, {'qty': '1 cup', 'name': 'pickled carrots'}, {'qty': '1 bunch', 'name': 'fresh cilantro'}, {'qty': '2 whole', 'name': 'jalapeño peppers'}, {'qty': '3 tbsp', 'name': 'soy sauce'}]
---
Sushi Bake with Salmon and Shrimp
<class 'list'>
[{'qty': '2 cups', 'name': 'sushi rice'}, {'qty': '1 lb', 'name': 'salmon fillet'}, {'qty': '8 oz', 'name': 'cooked shrimp'}, {'qty': '8 oz', 'name': 'cream cheese'}, {'qty': '1/2 cup', 'name': 'mayonnaise'}, {'qty': '2 tbsp', 'name': 'sriracha sauce'}, {'qty': '4 sheets', 'name': 'nori sheets'}, {'qty': '2 tbsp', 'name': 'sesame seeds'}, {'qty': '3 stalks', 'name': 'green onions'}]
---
Chicken Tikka Masala
<class 'list'>
[{'qty': '2 lbs', 'name': 'boneless chicken'}, {'qty': '1 cup', 'name': 'plain yogu

In [12]:
def meal_to_text(meal):
    """
    Converts a Firestore meal document into a single descriptive text string.
    This is what gets embedded — the richer the text, the better the semantic search.
    """
    parts = []
    
    if meal.get("name"):
        parts.append(f"Meal: {meal['name']}")
    if meal.get("cuisine"):
        parts.append(f"Cuisine: {meal['cuisine']}")
    if meal.get("protein"):
        parts.append(f"Protein: {meal['protein']}")
    if meal.get("ingredients"):
        ingredients = meal["ingredients"]
        if isinstance(ingredients, list):
            ingredient_parts = []
            for item in ingredients:
                if isinstance(item, dict):
                    # Pull whatever name field your dicts use
                    name = item.get("name") or item.get("ingredient") or str(item)
                    ingredient_parts.append(name)
                else:
                    ingredient_parts.append(str(item))
            parts.append(f"Ingredients: {', '.join(ingredient_parts)}")
    else:
        parts.append(f"Ingredients: {ingredients}")
    if meal.get("notes") or meal.get("description"):
        notes = meal.get("notes") or meal.get("description")
        parts.append(f"Notes: {notes}")
    if meal.get("tags"):
        parts.append(f"Tags: {', '.join(meal['tags'])}")
    
    return ". ".join(parts)

# Preview what a chunk looks like
for meal in meals[:3]:
    print(meal_to_text(meal))
    print("---")

Meal: Bahn Mi Subs. Cuisine: Vietnamese. Ingredients: chicken breast, Vietnamese baguettes, mayonnaise, cucumber, pickled daikon radish, pickled carrots, fresh cilantro, jalapeño peppers, soy sauce
---
Meal: Sushi Bake with Salmon and Shrimp. Cuisine: Japanese. Ingredients: sushi rice, salmon fillet, cooked shrimp, cream cheese, mayonnaise, sriracha sauce, nori sheets, sesame seeds, green onions
---
Meal: Chicken Tikka Masala. Cuisine: Indian. Ingredients: boneless chicken, plain yogurt, heavy cream, crushed tomatoes, onion, garlic cloves, fresh ginger, garam masala, turmeric powder, vegetable oil
---


In [13]:
def get_embedding(text):
    """
    Calls Anthropic's embedding endpoint to convert text into a vector.
    A vector is just a list of ~1500 floats that encode the semantic meaning of the text.
    """
    response = client.messages.create(
        model="claude-sonnet-4-6",  # or use the embeddings-specific model when available
        max_tokens=10,
        system="Return only the word OK.",
        messages=[{"role": "user", "content": text}]
    )
    # NOTE: Anthropic's dedicated embeddings endpoint uses a different call.
    # We'll use their voyage embeddings via the API below.
    return response

# Actually — Anthropic uses Voyage AI for embeddings. Let's use that correctly:
import voyageai

voyage_client = voyageai.Client(api_key=secrets["VOYAGE_API_KEY"])  # uses VOYAGE_API_KEY env var

def get_embedding(text):
    result = voyage_client.embed([text], model="voyage-2")
    return result.embeddings[0]

# Test it
test_vector = get_embedding("high protein chicken meal")
print(f"Vector length: {len(test_vector)}")
print(f"First 5 values: {test_vector[:5]}")

Vector length: 1024
First 5 values: [-0.004289522767066956, 0.011098645627498627, 0.017334984615445137, 0.0391211174428463, -0.05095440894365311]


In [22]:
# ChromaDB stores vectors locally in a folder called "ourbrain_chroma"
chroma_client = chromadb.PersistentClient(path="./ourbrain_chroma")

# Delete old collection if rebuilding (safe to re-run)
try:
    chroma_client.delete_collection("meals")
    print("Deleted existing meals collection.")
except:
    pass

collection = chroma_client.create_collection(
    name="meals",
    metadata={"hnsw:space": "cosine"}  # cosine similarity = best for semantic search
)

print("ChromaDB collection created.")

Deleted existing meals collection.
ChromaDB collection created.


In [23]:
print(f"Meals in memory: {len(meals)}")
print(meals[0].get("name"))

Meals in memory: 11
Bahn Mi Subs


In [24]:
def embed_and_store_meals(meals):
    documents = []
    embeddings = []
    metadatas = []
    ids = []
    
    for i, meal in enumerate(meals):
        text = meal_to_text(meal)
        vector = get_embedding(text)
        
        documents.append(text)
        embeddings.append(vector)
        metadatas.append({
            "name": meal.get("name", ""),
            "cuisine": meal.get("cuisine", ""),
            "protein": meal.get("protein", ""),
            "meal_id": meal.get("id", str(i))
        })
        ids.append(f"meal_{meal.get('id', str(i))}")
        
        print(f"Embedded: {meal.get('name', 'unnamed')}")
    
    collection.add(
        documents=documents,
        embeddings=embeddings,
        metadatas=metadatas,
        ids=ids
    )
    
    print(f"\nStored {len(meals)} meals in ChromaDB.")

embed_and_store_meals(meals)

Embedded: Bahn Mi Subs
Embedded: Sushi Bake with Salmon and Shrimp
Embedded: Chicken Tikka Masala
Embedded: Ground Beef Shawarma
Embedded: Chicken Alfredo with Broccoli
Embedded: Greek Yogurt Pancakes
Embedded: Bibimbap
Embedded: Pesto chicken pasta
Embedded: Chicken quesadillas
Embedded: Salmon and Roasted Veggies
Embedded: Greek yogurt chocolate muffins

Stored 11 meals in ChromaDB.


In [25]:
def semantic_search(query, n_results=3):
    query_vector = get_embedding(query)
    
    results = collection.query(
        query_embeddings=[query_vector],
        n_results=n_results
    )
    
    print(f"Query: '{query}'\n")
    for i, doc in enumerate(results["documents"][0]):
        score = results["distances"][0][i]
        name = results["metadatas"][0][i].get("name", "unknown")
        print(f"Result {i+1}: {name} (distance: {score:.4f})")
        print(f"  {doc[:120]}...")
        print()

# Test with a structured query (tools can also answer this)
semantic_search("high protein meals")

# Test with a fuzzy query (this is where RAG earns its place)
semantic_search("something light but satisfying")
semantic_search("quick weeknight Asian dinner")

Query: 'high protein meals'

Result 1: Bahn Mi Subs (distance: 0.2479)
  Meal: Bahn Mi Subs. Cuisine: Vietnamese. Ingredients: chicken breast, Vietnamese baguettes, mayonnaise, cucumber, pickle...

Result 2: Bibimbap (distance: 0.2551)
  Meal: Bibimbap. Cuisine: Asian. Ingredients: cooked white rice, beef bulgogi or marinated beef strips, spinach, bean spr...

Result 3: Chicken Alfredo with Broccoli (distance: 0.2680)
  Meal: Chicken Alfredo with Broccoli. Cuisine: Italian. Ingredients: boneless chicken breast, fettuccine pasta, fresh bro...

Query: 'something light but satisfying'

Result 1: Bahn Mi Subs (distance: 0.3195)
  Meal: Bahn Mi Subs. Cuisine: Vietnamese. Ingredients: chicken breast, Vietnamese baguettes, mayonnaise, cucumber, pickle...

Result 2: Bibimbap (distance: 0.3218)
  Meal: Bibimbap. Cuisine: Asian. Ingredients: cooked white rice, beef bulgogi or marinated beef strips, spinach, bean spr...

Result 3: Chicken Tikka Masala (distance: 0.3348)
  Meal: Chicken Tikka Masa

In [26]:
import os
print(os.getcwd())

/Users/sagar/dev/TheBrain/notebooks
